# Xây Dựng & Chuẩn Hóa Bảng Nguồn Thu Thập: `job_sources`

Notebook này tập trung xây dựng bảng danh mục nguồn thu thập dữ liệu **`job_sources`** trong kiến trúc Silver Layer:
- Đóng vai trò là bảng chiều (Dimension Table) quản lý các kênh/sàn tuyển dụng được thu thập (TopCV, TopDev, VietnamWorks).
- Cung cấp khóa ngoại `source_id` cho các bảng nghiệp vụ và quan sát dữ liệu: `jobs.primary_source_id`, `job_source_records.source_id`, `job_snapshots.source_id`, và `dq_results.source_id`.
- Quản lý trạng thái kích hoạt (`is_active`), đường dẫn gốc (`base_url`) và mã định danh ngắn (`source_code`).
- Kiểm tra tính toàn vẹn ràng buộc DDL PostgreSQL (khóa chính, unique constraints).

In [1]:
import os
import json
from collections import Counter
from datetime import datetime
import pandas as pd
import numpy as np

# Cấu hình hiển thị pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
print('Môi trường xử lý bảng job_sources đã sẵn sàng!')

Môi trường xử lý bảng job_sources đã sẵn sàng!


## 1. Khám Phá & Phân Tích Nguồn Dữ Liệu Từ Tầng Bronze

In [2]:
listings_path = os.path.abspath('../../data/bronze/bronze_listings_master.json')

with open(listings_path, 'r', encoding='utf-8') as f:
    bronze_listings = json.load(f)

source_counts = Counter(l.get('source') for l in bronze_listings)
total_listings = len(bronze_listings)

source_analysis = []
for src, cnt in source_counts.items():
    sample_urls = [l.get('job_url', '') for l in bronze_listings if l.get('source') == src and l.get('job_url')]
    sample_url = sample_urls[0] if sample_urls else ''
    source_analysis.append({
        'Nguồn (source)': src,
        'Số lượng tin': cnt,
        'Tỷ lệ đóng góp (%)': round(cnt / total_listings * 100, 2),
        'Ví dụ job_url': sample_url[:80] + '...'
    })

df_source_stat = pd.DataFrame(source_analysis).sort_values(by='Số lượng tin', ascending=False)
print('--- THỐNG KÊ PHÂN BỐ NGUỒN THU THẬP TRÊN DỮ LIỆU BRONZE (2,327 TIN) ---')
print(df_source_stat.to_string(index=False))

--- THỐNG KÊ PHÂN BỐ NGUỒN THU THẬP TRÊN DỮ LIỆU BRONZE (2,327 TIN) ---
Nguồn (source)  Số lượng tin  Tỷ lệ đóng góp (%)                                                                       Ví dụ job_url
         topcv          1161               49.89 https://www.topcv.vn/viec-lam/full-stack-web-engineer-intern-fresher-ai-coding-p...
  vietnamworks          1046               44.95 https://www.vietnamworks.com/senior-allegro-pcb-layout-design-engineer-179074942...
        topdev           120                5.16    https://topdev.vn/viec-lam/network-engineer-cong-ty-tnhh-airquay-vina-2137766...


## 2. Khởi Tạo & Chuẩn Hóa Bảng `job_sources` Theo DDL PostgreSQL

Định nghĩa danh mục nguồn thu thập với 3 nguồn hiện hữu (TopCV, TopDev, VietnamWorks) và 1 nguồn mở rộng tiềm năng (ITViec).

In [3]:
now_iso = datetime.now().isoformat()

source_definitions = [
    {
        'source_id': 1,
        'source_name': 'TopCV',
        'source_code': 'topcv',
        'base_url': 'https://www.topcv.vn',
        'is_active': True,
        'created_at': now_iso,
        'updated_at': now_iso
    },
    {
        'source_id': 2,
        'source_name': 'TopDev',
        'source_code': 'topdev',
        'base_url': 'https://topdev.vn',
        'is_active': True,
        'created_at': now_iso,
        'updated_at': now_iso
    },
    {
        'source_id': 3,
        'source_name': 'VietnamWorks',
        'source_code': 'vietnamworks',
        'base_url': 'https://www.vietnamworks.com',
        'is_active': True,
        'created_at': now_iso,
        'updated_at': now_iso
    },
    {
        'source_id': 4,
        'source_name': 'ITViec',
        'source_code': 'itviec',
        'base_url': 'https://itviec.com',
        'is_active': False,
        'created_at': now_iso,
        'updated_at': now_iso
    }
]

df_job_sources = pd.DataFrame(source_definitions)

print('Khởi tạo bảng job_sources thành công:')
print(f'  - Tổng số nguồn thu thập được định nghĩa: {len(df_job_sources)}')
print(f'  - Số nguồn đang hoạt động (is_active = True): {(df_job_sources["is_active"] == True).sum()}')
print(f'  - Tính duy nhất của khóa chính source_id: {df_job_sources["source_id"].is_unique}')
print(f'  - Tính duy nhất của source_name: {df_job_sources["source_name"].is_unique}')
print(f'  - Tính duy nhất của source_code: {df_job_sources["source_code"].is_unique}')

Khởi tạo bảng job_sources thành công:
  - Tổng số nguồn thu thập được định nghĩa: 4
  - Số nguồn đang hoạt động (is_active = True): 3
  - Tính duy nhất của khóa chính source_id: True
  - Tính duy nhất của source_name: True
  - Tính duy nhất của source_code: True


## 3. Kiểm Tra Tính Đồng Bộ Với Bảng `jobs` & Các Bảng Khác

Xác thực rằng 100% giá trị `primary_source_id` trong bảng `jobs` đều tồn tại và hợp lệ trong bảng `job_sources`.

In [4]:
code_to_id = dict(zip(df_job_sources['source_code'], df_job_sources['source_id']))
mapped_ids = [code_to_id.get(l.get('source')) for l in bronze_listings]
valid_fk_count = sum(1 for sid in mapped_ids if sid in df_job_sources['source_id'].values)

print('--- KIỂM TRA ĐỒNG BỘ TOÀN VẸN KHÓA NGOẠI (FK VALIDATION) ---')
print(f'Tổng số tin tuyển dụng kiểm tra: {len(mapped_ids):,}')
print(f'Số tin có source_id hợp lệ trong job_sources: {valid_fk_count:,} / {len(mapped_ids)} ({valid_fk_count/len(mapped_ids)*100:.1f}%)')
print('Trạng thái: 100% khóa ngoại tham chiếu hợp lệ!')

--- KIỂM TRA ĐỒNG BỘ TOÀN VẸN KHÓA NGOẠI (FK VALIDATION) ---
Tổng số tin tuyển dụng kiểm tra: 2,327
Số tin có source_id hợp lệ trong job_sources: 2,327 / 2327 (100.0%)
Trạng thái: 100% khóa ngoại tham chiếu hợp lệ!


## 4. Đánh Giá Cấu Trúc Bảng & Xem Trước Bảng `job_sources`

In [5]:
print('--- 1. CẤU TRÚC BẢNG job_sources (INFO) ---')
print(df_job_sources.info())

print('\n--- 2. TOÀN BỘ DỮ LIỆU BẢNG job_sources (CHUẨN POSTGRESQL) ---')
print(df_job_sources[['source_id', 'source_name', 'source_code', 'base_url', 'is_active', 'created_at']].to_string(index=False))

--- 1. CẤU TRÚC BẢNG job_sources (INFO) ---
<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   source_id    4 non-null      int64
 1   source_name  4 non-null      str  
 2   source_code  4 non-null      str  
 3   base_url     4 non-null      str  
 4   is_active    4 non-null      bool 
 5   created_at   4 non-null      str  
 6   updated_at   4 non-null      str  
dtypes: bool(1), int64(1), str(5)
memory usage: 677.0 bytes
None

--- 2. TOÀN BỘ DỮ LIỆU BẢNG job_sources (CHUẨN POSTGRESQL) ---
 source_id  source_name  source_code                     base_url  is_active                 created_at
         1        TopCV        topcv         https://www.topcv.vn       True 2026-10-13T12:17:24.588056
         2       TopDev       topdev            https://topdev.vn       True 2026-10-13T12:17:24.588056
         3 VietnamWorks vietnamworks https://www.vietnamworks.com 